# Per-gene dose-response curve comparison

Generalized, dataset-agnostic version of `GEX_comp_Doming_Morris.ipynb` -- compares two fitted bayesDREAM models across all overlapping trans genes, one 2x2 panel per gene:

- top-left: dataset A data + dataset A curve (standalone, with param markers)
- top-right: dataset B data + dataset B curve (standalone, with param markers)
- bottom-left: dataset A data + dataset A curve + dataset B curve overlaid
- bottom-right: dataset B data + dataset B curve + dataset A curve overlaid

plus one guide-density panel (log2FC(x_true) by guide/cell_line) shared across all genes for a given cis gene.

**Prerequisite:** for each (dataset, cis_gene) you want here, `save_model_for_plotting()` (see `save_for_plotting.py` at the repo root) must already have been run once in the original fitting session, with its output directory registered in `comparative/datasets.py`'s `DatasetSpec.save_for_plotting_dir_fn`. This does a full model reload, so it's only meant for a bounded number of genes (Domingo's ~91 shared trans genes is fine; don't point this at Morris/Replogle transcriptome-wide -- use `trans_param_comparison.ipynb` for that instead).

In [ ]:
# run "pip install ipython-autotime" in your conda env
%load_ext autotime

import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch

from comparative.datasets import DOMINGO, MORRIS, REPLOGLE, DATASET_COLORS
from comparative.dose_response_panels import compare_pair, load_model_for_plotting

## Config

In [ ]:
deviceno = 2
DEVICE = f'cuda:{deviceno}' if torch.cuda.is_available() else 'cpu'

# Which two datasets + which cis gene. Both datasets need save_model_for_plotting()
# output already on disk for this gene (see comparative/datasets.py).
SPEC_A = DOMINGO
SPEC_B = MORRIS
CIS_GENE = 'GFI1B'

PLOT_DIR = './dose_response_comparison_plots'

# Only feasible for a small gene set (full model reload + one panel per gene).
# Leave as None to auto-use every trans gene shared by both datasets'
# trans_feature_summary CSVs (fine for Domingo's ~91-gene panel; for a
# transcriptome-wide dataset pair, pass an explicit short list here instead).
GENES = None

# Dataset-identity curve colours (Domingo/Morris/Replogle) -- distinct from
# both the steelblue/tomato cell-line data colours and each other. Each
# DatasetSpec already carries its own `.color`; DATASET_COLORS is here only
# if you want to override one for this notebook without editing datasets.py.
print(DATASET_COLORS)

# Standalone panels show fitted-parameter markers (EC50/inflection lines) by
# default. Turn off if they read as confusing next to the dataset-colour
# curves -- overlay panels never show markers regardless of this flag.
SHOW_PARAM_MARKERS = True

## Run

`compare_pair` loads both models, summarises both trans fits, finds (or uses `GENES` as) the shared trans gene set, and writes one panel PNG per gene plus one guide-density panel into `PLOT_DIR`.

In [ ]:
plotted = compare_pair(
    SPEC_A, SPEC_B, CIS_GENE,
    out_dir=PLOT_DIR,
    genes=GENES,
    show_param_markers=SHOW_PARAM_MARKERS,
    device=DEVICE,
)
print(f'Plotted {len(plotted)} genes: {plotted[:10]}{"..." if len(plotted) > 10 else ""}')

## Inspect a single panel inline

Handy for iterating on plot styling without re-running the whole loop above -- reload the two models once, then call `make_panel` directly for one gene at a time.

In [ ]:
from comparative.dose_response_panels import make_panel, resolve_sum_factor_col, allsig_copy
import matplotlib.pyplot as plt

model_a = load_model_for_plotting(SPEC_A, CIS_GENE, device=DEVICE)
model_b = load_model_for_plotting(SPEC_B, CIS_GENE, device=DEVICE)
sfcol_a, sfcol_b = resolve_sum_factor_col(SPEC_A, model_a), resolve_sum_factor_col(SPEC_B, model_b)
summary_a = model_a.save_trans_summary(compute_lfc_ci=False)
summary_b = model_b.save_trans_summary(compute_lfc_ci=False, compute_derivative_roots=False)
summary_a_allsig, summary_b_allsig = allsig_copy(summary_a), allsig_copy(summary_b)

In [ ]:
GOI = plotted[0] if plotted else None
fig, unified_x = make_panel(
    GOI, SPEC_A, model_a, summary_a_allsig, sfcol_a, SPEC_B, model_b, summary_b_allsig, sfcol_b,
    cis_gene=CIS_GENE, show_param_markers=SHOW_PARAM_MARKERS,
)
plt.show()

## Three-way comparison for one gene

If all three datasets have `save_model_for_plotting()` output for the same cis gene, compare every pair (each `compare_pair` call writes its own panels + density plot).

In [ ]:
from itertools import combinations

THREE_WAY_CIS_GENE = 'GFI1B'
for a, b in combinations([DOMINGO, MORRIS, REPLOGLE], 2):
    try:
        compare_pair(a, b, THREE_WAY_CIS_GENE, out_dir=PLOT_DIR, show_param_markers=SHOW_PARAM_MARKERS, device=DEVICE)
    except FileNotFoundError as e:
        print(f'[skip {a.name} vs {b.name}] {e}')